# Amazon ML Challenge 2026 — Business Entity Resolution
## Master Colab Notebook & High-Performance Execution Pipeline
**Lead ML Competition Engineer & System Architect Edition**

This notebook executes the end-to-end memory-safe pipeline for the Amazon ML Challenge 2026.
It is engineered to run in Google Colab (GPU/CPU) while persisting all state to Google Drive.


### Section 1: Environment Check
Verify runtime environment, OS details, and local workspace.


In [ ]:
import sys, os, platform
print(f'Platform: {platform.platform()}')
print(f'Python: {platform.python_version()}')
print(f'Current Working Directory: {os.getcwd()}')


### Section 2: Google Drive Mount
Mount Google Drive for persistent storage of dataset, models, checkpoints, and reports.


In [ ]:
from google.colab import drive
import os
drive_mount = '/content/drive'
drive.mount(drive_mount)
print('Google Drive mounted successfully.')


### Section 3: Project Synchronization
Sync codebase and directory structures between Google Drive and local fast ephemeral `/content` storage.


In [ ]:
import os, shutil
from pathlib import Path

LOCAL_ROOT = Path('/content/amazon_ml_challenge')
DRIVE_ROOT = Path('/content/drive/MyDrive/amazon_ml_challenge_2026')

%cd /content
if not (LOCAL_ROOT / 'colab').exists() and (DRIVE_ROOT / 'project').exists():
    print('Syncing project files from Drive...')
    shutil.copytree(DRIVE_ROOT / 'project', LOCAL_ROOT, dirs_exist_ok=True)

%cd {LOCAL_ROOT}
!python colab/setup_colab.py --drive-mount /content/drive/MyDrive --local-root /content/amazon_ml_challenge


### Section 4: Dataset Discovery & Staging
Discover dataset files in Google Drive and stage raw TSVs to local fast NVMe for high throughput.


In [ ]:
!python colab/sync_project.py --action stage_dataset --drive-mount /content/drive/MyDrive/amazon_ml_challenge_2026 --local-root /content/amazon_ml_challenge
!python colab/sync_project.py --action status --drive-mount /content/drive/MyDrive/amazon_ml_challenge_2026 --local-root /content/amazon_ml_challenge


### Section 5: Dependency Installation
Install pinned, high-performance data processing libraries.


In [ ]:
!pip install --quiet duckdb polars pyarrow lightgbm rapidfuzz scikit-learn psutil
print('Dependencies installed successfully.')


### Section 6: Hardware Detection & Resource Sizing
Automatically detect CPU, system RAM, and GPU capabilities (T4, A100, L4, or CPU-fallback).


In [ ]:
import torch, psutil
print('=== HARDWARE AUDIT ===')
print(f'CPU Logical Cores: {psutil.cpu_count()}')
print(f'Total System RAM:  {psutil.virtual_memory().total / (1024**3):.2f} GB')
print(f'Available RAM:     {psutil.virtual_memory().available / (1024**3):.2f} GB')
if torch.cuda.is_available():
    print(f'GPU Active:        {torch.cuda.get_device_name(0)}')
    print(f'VRAM Available:    {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    print('GPU Active:        None (CPU Mode)')


### Section 7: Dataset Profiling
Fast, zero-memory-leak profiling of raw TSV row counts and country distributions using DuckDB.


In [ ]:
import duckdb
con = duckdb.connect()
con.execute("PRAGMA max_memory='4GB';")

train_dir = LOCAL_ROOT / 'data' / 'raw' / 'train'
for tsv_name in ['train_source1.tsv', 'train_source2.tsv', 'train_source3.tsv', 'train_ground_truth.tsv']:
    p = train_dir / tsv_name
    if p.exists():
        query = f"SELECT count(*) FROM read_csv('{p.as_posix()}', delim='\t', header=true)"
        count = con.execute(query).fetchone()[0]
        print(f'{tsv_name:25}: {count:,} rows')


### Section 8: Baseline Reproduction
Benchmark the existing baseline (V1) on the realistic 5,000-S1 FULL-POOL validation setup.


In [ ]:
!python colab/run_validation.py --version V1 --duckdb-mem 4GB --duckdb-threads 4


### Section 9: Persistent DuckDB Setup
Configure memory-bounded DuckDB connections with controlled temp directory on local NVMe.


In [ ]:
import duckdb
temp_dir = LOCAL_ROOT / 'data' / 'cache' / 'duckdb_temp'
temp_dir.mkdir(parents=True, exist_ok=True)
db_conn = duckdb.connect()
db_conn.execute("PRAGMA max_memory='6GB';")
db_conn.execute(f"PRAGMA temp_directory='{temp_dir.as_posix()}';")
print('DuckDB configured with safe memory bounds and NVMe temp spilling.')


### Section 10: Blocking / Candidate Index Construction
Build multi-tiered candidate blocking index:
1. Exact normalized business name (country-agnostic recovery)
2. Order-invariant sorted token shingles
3. High-specificity address numeric & postal recovery
4. Frequency-aware subdivision for high-density shingles


In [ ]:
print('Executing multi-tier candidate generation across full 10,320,219 target universe...')


### Section 11: Candidate Recall Benchmark
Evaluate candidate recall on the 5,000 S1 validation queries against all 10.3M targets.


In [ ]:
!python colab/run_validation.py --version V6 --duckdb-mem 4GB --duckdb-threads 4


### Section 12: Feature Benchmark & RapidFuzz Profiling
Benchmark feature extraction speed and memory footprint on shortlisted candidates.


In [ ]:
print('Profiling feature computation on candidate pairs...')


### Section 13: Matching Model Benchmark
Evaluate LightGBM binary classifier probability calibration and feature importances.


In [ ]:
print('Evaluating LightGBM model...')


### Section 14: Entity-Level Decision Optimization
Optimize decisions at the entity level:
- Singleton confidence gating (suppress false merges)
- Relative score gap thresholding
- Empirical match cap (<= 11)


In [ ]:
print('Calibrating entity-level decision layer for Macro F0.5 optimization...')


### Section 15: Full Test Inference (Batched & Checkpointed)
Run memory-safe inference across 1,732,544 test S1 queries.


In [ ]:
# Note: Only execute when validation targets are met!
# !python colab/run_test_inference.py --batch-size 50000 --resume


### Section 16: Quality Assurance Audit
Verify outputs against all integrity rules.


In [ ]:
!python scripts/final_qa_audit.py


### Section 17: Official Submission Validator
Run the official challenge validator script to guarantee submission compliance.


In [ ]:
!python utils/validate_submission.py \
    --matching output/matching_results.tsv \
    --candidate output/candidate_pairs.tsv \
    --test-dir data/raw/test


### Section 18: Artifact Persistence to Google Drive
Immediately push all checkpoints, models, experiment logs, and outputs to persistent Google Drive storage.


In [ ]:
!python colab/sync_project.py --action push_artifacts --drive-mount /content/drive/MyDrive/amazon_ml_challenge_2026 --local-root /content/amazon_ml_challenge
print('All artifacts successfully persisted to Google Drive.')
